<a href="https://colab.research.google.com/github/tsukki8/ds2002-fa26/blob/main/notebooks/05-cleaning-clinic/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [1]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [2]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [3]:
# TODO
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)

print("\nNull count per column:")
print(df.isna().sum())

print("\nExact duplicate rows:", df.duplicated().sum())

Shape: (8, 6)

Dtypes:
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

Null count per column:
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

Exact duplicate rows: 1


**What is wrong with this data?** List at least five specific problems:

1. a duplicate row
2. a null item
3. a null qty
4. a null ts
5. price should be stored as float and not object

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [4]:
removed = df.duplicated().sum()
clean = df.drop_duplicates().copy()

# TODO: log('duplicates', 'dropped exact duplicate rows', removed)
log(
    'duplicates',
    'dropped exact duplicate rows',
    removed
)

print("Rows after removing duplicates:", len(clean))

[duplicates] dropped exact duplicate rows (1 row(s))
Rows after removing duplicates: 7


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [5]:
clean['price'] = (
    clean['price']
    .astype(str)
    .str.strip()
    .str.replace('$', '', regex=False)
    .astype(float)
)

assert clean['price'].dtype == float

log(
    'price',
    'stripped dollar signs and whitespace, then converted price to float',
    len(clean)
)

[price] stripped dollar signs and whitespace, then converted price to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [7]:
# TODO: clean['qty'] = pd.to_numeric(...)

clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()
negative = (clean['qty'] < 0).sum()

# TODO: apply your decision, then log both separately
print("Missing quantities:", missing)
print("Negative quantities:", negative)

clean = clean.dropna(subset=['qty']).copy()

log(
    'quantity_missing',
    'dropped rows with missing quantity because revenue could not be calculated reliably',
    missing
)

clean = clean[clean['qty'] >= 0].copy()

log(
    'quantity_negative',
    'dropped negative-quantity refund from demand data',
    negative
)

Missing quantities: 1
Negative quantities: 1
[quantity_missing] dropped rows with missing quantity because revenue could not be calculated reliably (1 row(s))
[quantity_negative] dropped negative-quantity refund from demand data (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [9]:
print('before:', sorted(clean['category'].unique()))

# TODO: lowercase, strip, remove punctuation
# TODO: CATEGORY_MAP = {...} for the judgment calls

# print('after: ', sorted(clean['category'].unique()))
clean['category'] = (
    clean['category']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]', '', regex=True)
)

CATEGORY_MAP = {
    'food': 'Food',
    'merch': 'Merch',
    'apparel': 'Merch',
    'raingear': 'RainGear'
}

clean['category'] = clean['category'].map(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))

categories_before = 4
categories_after = clean['category'].nunique()

log(
    'category',
    'normalized case/punctuation and mapped Apparel to Merch',
    categories_before - categories_after
)

before: ['apparel', 'food', 'merch', 'raingear']
after:  ['Food', 'Merch', 'RainGear']
[category] normalized case/punctuation and mapped Apparel to Merch (1 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [10]:
# TODO
clean['item'] = (
    clean['item']
    .fillna('Unknown Item')
    .str.strip()
    .str.lower()
    .str.replace(r'[^a-z0-9]', '', regex=True)
)

ITEM_MAP = {
    'cheeseburger': 'Cheeseburger',
    'foamfinger': 'Foam Finger',
    'uvatshirt': 'UVA T-Shirt',
    'rainponcho': 'Rain Poncho',
    'unknownitem': 'Unknown Item'
}

clean['item'] = clean['item'].map(ITEM_MAP)

log(
    'item',
    'normalized item spelling and replaced missing item with Unknown Item',
    1
)

print(clean[['item', 'category']])

[item] normalized item spelling and replaced missing item with Unknown Item (1 row(s))
           item  category
0  Cheeseburger      Food
2  Cheeseburger      Food
4   UVA T-Shirt     Merch
6   Rain Poncho  RainGear
7  Unknown Item     Merch


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [11]:
# TODO
clean['ts'] = pd.to_datetime(
    clean['ts'],
    errors='coerce'
)

failed_timestamps = clean['ts'].isna().sum()

print("Failed timestamps:", failed_timestamps)

clean['hour'] = clean['ts'].dt.hour

log(
    'timestamp',
    'parsed timestamps and converted invalid/missing values to NaT',
    failed_timestamps
)

Failed timestamps: 3
[timestamp] parsed timestamps and converted invalid/missing values to NaT (3 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [12]:
# TODO: assertions
assert len(clean) == 5
assert clean['price'].dtype == float
assert pd.api.types.is_numeric_dtype(clean['qty'])
assert (clean['qty'] >= 0).all()
assert clean['category'].notna().all()
assert clean['item'].notna().all()
assert 'hour' in clean.columns
assert clean['ts'].dtype == 'datetime64[ns]'

# TODO: clean['revenue'] = ...
# TODO: print rows, units, revenue, distinct categories
clean['revenue'] = clean['qty'] * clean['price']

print("Rows:", len(clean))
print("Total units:", clean['qty'].sum())
print("Total revenue:", clean['revenue'].sum())
print("Distinct categories:", clean['category'].nunique())

Rows: 5
Total units: 10.0
Total revenue: 106.5
Distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [13]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,"stripped dollar signs and whitespace, then con...",7
2,quantity_missing,dropped rows with missing quantity because rev...,1
3,quantity_negative,dropped negative-quantity refund from demand data,1
4,category,normalized case/punctuation and mapped Apparel...,1
5,item,normalized item spelling and replaced missing ...,1
6,timestamp,parsed timestamps and converted invalid/missin...,3


**The decision that mattered most:** dropping negative-quantity refunds

**Revenue with it:** 106.50  **Revenue without it:** 88.50

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [14]:
# Checkpoint
rows_after = 5            # TODO
revenue_after = 106.5         # TODO
biggest_decision = 'dropping negative-quantity refunds'    # TODO: which choice moved the number most
revenue_other_way = 88.50     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 106.5
decision that mattered: dropping negative-quantity refunds
revenue the other way: 88.5
